# Experiment: Compare metric depth models on the same nine frames

**Question:** Which model provides the best depth quality for the time and GPU resources needed by COMP0227 CW1?

This compares **Depth Pro**, **Depth Anything V2 Small/Base/Large**, and **Depth Anything 3 Metric Large**. Each model sees the same three spread frames from Replica, TUM and KITTI. V2 uses its declared Hypersim checkpoint indoors and VKITTI checkpoint outdoors. Depth Pro and DA3 are measured separately in FP32 and FP16; V2 uses FP32. There are ten sequential case commands, with no concurrent models.

1. Open this notebook in [Colab](https://colab.research.google.com/).
2. Choose **Runtime > Change runtime type > GPU**, with **Runtime version 2026.07** (Python 3.12).
3. Run the cells in order, uploading **depth_model_comparison_inputs.zip** when prompted.
4. Download the results ZIP before ending the session. Keep the notebook private.

A Google account and available GPU allocation are needed. This notebook does not purchase compute or mount Drive. It stops before downloading weights if CUDA is unavailable; there is no CPU fallback. Downloads total about **6.9 GB** of weights, plus dependencies and three pinned source repositories.

This nine-frame sample is a model-selection pilot. It does not establish full-sequence accuracy, temporal consistency or SLAM performance. KITTI has no supplied depth GT, so its numerical comparison concerns speed; visual plausibility alone does not prove accuracy. GPU results must be measured on your runtime; none are supplied or claimed here.

References: [Colab runtime versions](https://research.google.com/colaboratory/runtime-version-faq.html), [Depth Pro](https://github.com/apple-aiml-research/ml-depth-pro), [V2 metric models](https://github.com/DepthAnything/Depth-Anything-V2/tree/main/metric_depth), [DA3 metric conversion](https://github.com/ByteDance-Seed/Depth-Anything-3#-faq).

## 1. Check Python and CUDA before downloading anything

Use runtime 2026.07. Numerical packages are probed in fresh processes, leaving the notebook kernel untouched. Strict FP32 cases disable NVIDIA TF32 through the child-process environment; FP16 is an explicitly separate case.

In [ ]:
import datetime
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys

if sys.version_info[:2] != (3, 12):
    raise RuntimeError('Select Runtime > Change runtime type > Runtime version 2026.07, then reconnect.')
probe_code = """import json, torch
available = torch.cuda.is_available()
print(json.dumps({'cuda_available': available, 'torch': torch.__version__, 'cuda_version': torch.version.cuda,
                  'gpu': torch.cuda.get_device_name(0) if available else None,
                  'gpu_memory_gib': torch.cuda.get_device_properties(0).total_memory / 2**30 if available else None}))
"""
gpu_info = json.loads(subprocess.check_output([sys.executable, '-c', probe_code], text=True))
if not gpu_info['cuda_available']:
    raise RuntimeError('No CUDA GPU allocation. Stop here and try a GPU runtime later; do not run the cases on CPU.')
if shutil.disk_usage('/content').free < 12 * 1024**3:
    raise RuntimeError('At least 12 GiB of free temporary disk is needed for weights, environments and outputs.')
run_env = {**os.environ, 'PYTHONDONTWRITEBYTECODE': '1', 'NVIDIA_TF32_OVERRIDE': '0',
           'MPLBACKEND': 'Agg', 'TOKENIZERS_PARALLELISM': 'false'}
nvidia_info = subprocess.check_output(['nvidia-smi'], text=True)
print(json.dumps(gpu_info, indent=2))
print(nvidia_info)

## 2. Upload the nine-frame input bundle

Upload the small **depth_model_comparison_inputs.zip**, containing only selected images, available GT, calibration and scripts. Frame indices and timestamps are retained explicitly; do not run the full-sequence loader on these sparse Replica images.

In [ ]:
from collections import Counter
from google.colab import files
import io
import stat
import zipfile

uploaded = files.upload()
archives = [name for name in uploaded if name.endswith('.zip')]
if len(archives) != 1:
    raise ValueError('Upload exactly one ZIP: depth_model_comparison_inputs.zip')
workspace = Path('/content/cw1_depth_model_comparison_run')
workspace.mkdir(parents=True, exist_ok=True)
input_archive_sha256 = hashlib.sha256(uploaded[archives[0]]).hexdigest()
with zipfile.ZipFile(io.BytesIO(uploaded[archives[0]])) as archive:
    entries = archive.infolist()
    if sum(info.file_size for info in entries) > 50 * 1024**2:
        raise ValueError('The selected-input bundle should expand to less than 50 MiB')
    if len({info.filename for info in entries}) != len(entries):
        raise ValueError('Duplicate archive paths are not accepted')
    for info in entries:
        member = Path(info.filename)
        destination = (workspace / member).resolve()
        if (member.is_absolute() or '..' in member.parts or not member.parts
                or member.parts[0] != 'cw1_depth_model_comparison'
                or not destination.is_relative_to(workspace.resolve())
                or stat.S_ISLNK(info.external_attr >> 16)):
            raise ValueError(f'Unexpected archive member: {info.filename}')
    archive.extractall(workspace)
project = workspace / 'cw1_depth_model_comparison'
manifest_path = project / 'sample_manifest.json'
expected_manifest_sha256 = '1345e08be0c92f6a125b7b9271e28a8c0d93ea7f2d9481849ddcc349b51ec8f0'
if hashlib.sha256(manifest_path.read_bytes()).hexdigest() != expected_manifest_sha256:
    raise ValueError('The uploaded selection differs from the shared CPU/GPU comparison manifest')
sample_manifest = json.loads(manifest_path.read_text())
expected_indices = {'replica_room0': [200, 1000, 1799], 'tum_fr3': [258, 1292, 2326], 'kitti06': [110, 550, 990]}
if Counter(row['sequence'] for row in sample_manifest) != Counter({name: 3 for name in expected_indices}):
    raise ValueError('Expected exactly three frames from each sequence')
for name, indices in expected_indices.items():
    if [row['index'] for row in sample_manifest if row['sequence'] == name] != indices:
        raise ValueError(f'Original frame indices differ for {name}')
for name in ('benchmark_models.py', 'model_adapters.py', 'da3_adapter.py', 'predict_depth.py', 'summarize_model_benchmarks.py'):
    if not (project / 'scripts/part2' / name).is_file():
        raise FileNotFoundError(f'Missing {name}; regenerate the current comparison bundle')
results_root = project / 'results/model_comparison_gpu'
results_root.mkdir(parents=True, exist_ok=True)
(results_root / 'sample_manifest.json').write_bytes(manifest_path.read_bytes())
(results_root / 'nvidia_smi.txt').write_text(nvidia_info)
(results_root / 'runtime.json').write_text(json.dumps({
    'started_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'base_gpu': gpu_info,
    'input_archive_sha256': input_archive_sha256, 'sample_manifest_sha256': expected_manifest_sha256,
    'tf32_override': '0', 'selection': expected_indices,
}, indent=2))
print('Input bundle verified:', len(sample_manifest), 'frames; SHA256', input_archive_sha256)

## 3. Create a small isolated environment

The environment inherits Colab's CUDA PyTorch/Torchvision. Only the numerical and model-support packages are installed into the venv. DA3's full application dependencies, xformers and Open3D are not installed; the adapter uses its metric network and official image processor directly.

In [ ]:
env_root = Path('/content/cw1_depth_model_comparison_env')
python = env_root / 'bin/python'
if not python.is_file():
    subprocess.run([sys.executable, '-m', 'venv', '--without-pip', '--system-site-packages', str(env_root)], check=True)
subprocess.run([str(python), '-m', 'pip', 'install', '--quiet', '--disable-pip-version-check',
                'numpy==1.26.4', 'opencv-python-headless==4.10.0.84', 'timm==1.0.30',
                'pillow_heif', 'scipy', 'pandas', 'pyyaml', 'matplotlib', 'einops',
                'omegaconf', 'addict', 'imageio', 'safetensors', 'huggingface-hub'], check=True)
env_gpu_info = json.loads(subprocess.check_output([str(python), '-c', probe_code], text=True, env=run_env))
if (not env_gpu_info['cuda_available'] or env_gpu_info['torch'] != gpu_info['torch']
        or env_gpu_info['cuda_version'] != gpu_info['cuda_version']):
    raise RuntimeError('CUDA/PyTorch changed unexpectedly; stop and inspect this environment')
subprocess.run([str(python), '-c',
                'import numpy, cv2, torch, torchvision; assert numpy.__version__ == "1.26.4"; '
                'print("NumPy", numpy.__version__, "OpenCV", cv2.__version__, "CUDA", torch.cuda.is_available())'],
               check=True, env=run_env)
freeze = subprocess.check_output([str(python), '-m', 'pip', 'freeze'], text=True)
(results_root / 'environment.txt').write_text(freeze)
(results_root / 'venv_gpu.json').write_text(json.dumps(env_gpu_info, indent=2))

## 4. Check out the pinned official source

Depth Pro is installed editable with `--no-deps`, preserving Colab's Torch. V2 and DA3 are loaded from their pinned source directories by the supplied adapters. These three source commits and all checkpoint checksums are recorded with the results.

In [ ]:
source_parent = Path('/content/cw1_depth_model_sources')
source_parent.mkdir(exist_ok=True)
source_specs = {
    'depth_pro': ('https://github.com/apple-aiml-research/ml-depth-pro.git', '9e65e4dbe9568d23c546fcec53302b10445e109e'),
    'dav2': ('https://github.com/DepthAnything/Depth-Anything-V2.git', 'a561b849ebae10a6f5ef49e26c83cbbcd36c71bf'),
    'da3': ('https://github.com/ByteDance-Seed/Depth-Anything-3.git', '3d835ec1a5802d64a8b8b15f817a1ab54809bfe4'),
}
source_roots = {}
for name, (url, commit) in source_specs.items():
    destination = source_parent / name
    if not (destination / '.git').is_dir():
        if destination.exists() and any(destination.iterdir()):
            raise RuntimeError(f'Unexpected nonempty source folder: {destination}')
        subprocess.run(['git', 'init', str(destination)], check=True)
        subprocess.run(['git', '-C', str(destination), 'remote', 'add', 'origin', url], check=True)
    present = subprocess.run(['git', '-C', str(destination), 'cat-file', '-e', commit + '^{commit}'],
                             stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0
    if not present:
        subprocess.run(['git', '-C', str(destination), 'fetch', '--depth', '1', 'origin', commit], check=True)
    subprocess.run(['git', '-C', str(destination), 'checkout', '--detach', commit], check=True)
    actual = subprocess.check_output(['git', '-C', str(destination), 'rev-parse', 'HEAD'], text=True).strip()
    if actual != commit:
        raise RuntimeError(f'Source revision mismatch for {name}')
    source_roots[name] = destination
subprocess.run([str(python), '-m', 'pip', 'install', '--quiet', '--no-deps', '-e', str(source_roots['depth_pro'])], check=True)
subprocess.run([str(python), '-c', 'import depth_pro; print("Pinned Depth Pro import passed")'], check=True, env=run_env)
(results_root / 'source_commits.json').write_text(json.dumps({name: {'url': url, 'commit': commit}
    for name, (url, commit) in source_specs.items()}, indent=2))

## 5. Download the official checkpoints with resumable transfers

The eight model-weight files total about 6.9 GB. Each Hugging Face URL uses an immutable revision; every download is checked against its expected byte count and SHA256. Interrupted `.partial` downloads can resume by rerunning this cell. A checksum failure stops before inference.

In [ ]:
weights_root = Path('/content/cw1_depth_model_weights')
weights_root.mkdir(exist_ok=True)
da3_revision = '4010e39f3634a45bc60553321fb49fb760bd594e'
download_specs = [
    ('depth_pro', 'https://ml-site.cdn-apple.com/models/depth-pro/depth_pro.pt',
     'depth_pro.pt', 1904446787, '3eb35ca68168ad3d14cb150f8947a4edf85589941661fdb2686259c80685c0ce'),
    ('da3_config', f'https://huggingface.co/depth-anything/DA3METRIC-LARGE/resolve/{da3_revision}/config.json',
     'da3metric-large/config.json', 847, 'a336f3e76fe375aaae17a9aed9130c9f2aa061535d317ec57dcb2f1f02e1dd53'),
    ('da3_weights', f'https://huggingface.co/depth-anything/DA3METRIC-LARGE/resolve/{da3_revision}/model.safetensors',
     'da3metric-large/model.safetensors', 1336734448, 'bbea5b0b3ee389849cffa7ddae89de064a90abd2b055fc5aa99aac68db324776'),
]
v2_specs = [
    ('Hypersim', 'Small', 'vits', '3bc65d4e14a6786a61acec16453c50e12bf5f338', 99222290, 'b782898d8a3e8be1f639de33837ed85e9b4b73e40f8f5e5cd99067588d722545'),
    ('Hypersim', 'Base', 'vitb', '4ebcd7fd0858cbb953451619ca943ea39ec08e00', 389965138, '9dc9e274c5eff55c6daf27b660c0ced0eca4e8593a6da90cdcb04d2b4d3f3fa2'),
    ('Hypersim', 'Large', 'vitl', '79720800638389a78b2defc92caa885104f69974', 1341401882, '6f82ff2bc543ac02ddff4aa31fa363676a8305dd3ccf04e80e2af115a044cb6d'),
    ('VKITTI', 'Small', 'vits', 'c725b8589bdf6ab04072cab74c0467830db80d6d', 99221808, '9203e538d35255c90dda4b7fedb47ff33fe725497bcca3b1e53b3a65ee63f0cb'),
    ('VKITTI', 'Base', 'vitb', 'e96fd0c8cedf1825a55b9c61309f754982948df0', 389964656, '4dad67a7cc10b462bca48e6b8569c762b8eb3c1adada170a3851a6d3ba37bb3e'),
    ('VKITTI', 'Large', 'vitl', '070e97e4b80e317ec1d03c19927304f4091a180b', 1341401064, '239b1054a369e66da2576e9a118d6d7c12d90dc8ebe609579a9a09cd8e05fe38'),
]
for domain, size, encoder, revision, expected_bytes, expected_sha in v2_specs:
    name = f'depth_anything_v2_metric_{domain.lower()}_{encoder}.pth'
    url = f'https://huggingface.co/depth-anything/Depth-Anything-V2-Metric-{domain}-{size}/resolve/{revision}/{name}'
    download_specs.append((f'dav2_{size.lower()}_{domain.lower()}', url, name, expected_bytes, expected_sha))

def digest_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(8 * 1024**2), b''):
            digest.update(block)
    return digest.hexdigest()

def download_checked(url, relative, expected_bytes, expected_sha):
    target = weights_root / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.is_file():
        candidate = target
    else:
        candidate = target.with_name(target.name + '.partial')
        if candidate.exists() and candidate.stat().st_size > expected_bytes:
            raise ValueError(f'Oversized partial download: {candidate}; inspect it before retrying')
        if not candidate.is_file() or candidate.stat().st_size != expected_bytes:
            subprocess.run(['curl', '-fL', '--retry', '3', '--continue-at', '-', url, '-o', str(candidate)], check=True)
    if candidate.stat().st_size != expected_bytes:
        raise ValueError(f'Incomplete {relative}; rerun this cell to resume')
    checksum = digest_file(candidate)
    if checksum != expected_sha:
        raise ValueError(f'SHA256 mismatch for {relative}; do not use this file')
    if candidate != target:
        candidate.replace(target)
    return target, checksum

checkpoint_paths, checkpoint_records = {}, []
for key, url, relative, expected_bytes, expected_sha in download_specs:
    target, checksum = download_checked(url, relative, expected_bytes, expected_sha)
    checkpoint_paths[key] = target
    checkpoint_records.append({'key': key, 'url': url, 'file': relative, 'bytes': expected_bytes, 'sha256': checksum})
    (results_root / 'checkpoints.json').write_text(json.dumps(checkpoint_records, indent=2))
    print('Verified', relative, f'({expected_bytes / 1024**2:.1f} MiB)', flush=True)
da3_config = json.loads(checkpoint_paths['da3_config'].read_text())
if da3_config.get('model_name', '').lower() != 'da3metric-large':
    raise ValueError('The DA3 checkpoint is not the required metric model')

## 6. Verify the portable images and depth associations

This reads all nine selected inputs without loading a model. It checks resolution against the manifest calibration and reports the original indices. Six frames have GT; KITTI has none. GT is used only to score predictions.

In [ ]:
inventory_code = r"""
import json, sys
from pathlib import Path
import cv2
sys.path.insert(0, 'scripts')
from part2.benchmark_models import load_samples
samples = load_samples(('replica_room0','tum_fr3','kitti06'), sample_manifest=Path(sys.argv[1]))
rows=[]
for sample in samples:
    frame,camera=sample['frame'],sample['camera']
    image=cv2.imread(str(frame.rgb))
    assert image is not None and image.shape[:2] == (camera.height,camera.width)
    if frame.depth_gt is not None:
        gt=cv2.imread(str(frame.depth_gt),cv2.IMREAD_UNCHANGED)
        assert gt is not None and gt.shape == image.shape[:2]
        assert abs(frame.depth_timestamp-frame.timestamp) <= .02
    rows.append({'sequence':sample['sequence'],'index':frame.index,'timestamp':frame.timestamp,
                 'rgb':frame.rgb.name,'gt_available':frame.depth_gt is not None,
                 'size':[camera.width,camera.height]})
assert len(rows)==9 and sum(row['gt_available'] for row in rows)==6
print(json.dumps(rows))
"""
inventory = json.loads(subprocess.check_output([str(python), '-c', inventory_code, str(manifest_path)],
                                               cwd=project, env=run_env, text=True))
(results_root / 'input_inventory.json').write_text(json.dumps(inventory, indent=2))
print(json.dumps(inventory, indent=2))

## 7. Declare the ten cases

Depth Pro and DA3 each run in FP32 and FP16 on all nine images. Each V2 size runs once indoors (Hypersim, six images) and once outdoors (VKITTI, three images), both in FP32. Their effective image sizes differ by their official preprocessing policies and are recorded; these are practical model configurations, not equal-resolution architecture ablations.

Each case has a separate results file and prediction directory. Setup and warmup are recorded separately. Timed inference includes preprocessing, synchronized network execution and postprocessing; it excludes image I/O, saving and GT scoring. Report failed cases and prediction coverage alongside errors.

In [ ]:
benchmark_root = results_root / 'benchmarks'
logs_root = results_root / 'logs'
benchmark_root.mkdir(exist_ok=True)
logs_root.mkdir(exist_ok=True)
cases = []
for model, source_key, checkpoint in (
    ('depth_pro', 'depth_pro', checkpoint_paths['depth_pro']),
    ('da3_metric_large', 'da3', checkpoint_paths['da3_weights'].parent),
):
    for precision in ('float32', 'float16'):
        cases.append({'label': f'spread_{model}_cuda_{precision}', 'model': model,
                      'precision': precision, 'source_root': str(source_roots[source_key]),
                      'checkpoint': str(checkpoint), 'sequences': ['replica_room0','tum_fr3','kitti06'],
                      'process_res': 504 if model == 'da3_metric_large' else None, 'domain': None})
for size in ('small','base','large'):
    for domain, sequences in (('hypersim',['replica_room0','tum_fr3']), ('vkitti',['kitti06'])):
        model = f'dav2_{size}'
        cases.append({'label': f'spread_{model}_{domain}_cuda_float32', 'model': model,
                      'precision': 'float32', 'source_root': str(source_roots['dav2']),
                      'checkpoint': str(checkpoint_paths[f'{model}_{domain}']),
                      'sequences': sequences, 'process_res': 518, 'domain': domain})
for case in cases:
    case['output'] = str(benchmark_root / (case['label'] + '.json'))
    case['pred_root'] = str(project / 'depth_maps/model_comparison_gpu' / case['label'])
assert len(cases) == 10 and len({case['label'] for case in cases}) == 10
(results_root / 'case_plan.json').write_text(json.dumps(cases, indent=2))
for case in cases:
    print(case['label'], len(case['sequences']) * 3, 'frames')

## 8. Run one case at a time; keep failures and progress

This cell can take several minutes. Every case runs in its own process, releasing model memory before the next case. Errors, including insufficient GPU memory, remain in the logs and results; the loop continues to the other declared cases. It never changes device, precision or resolution after a failure.

The harness checkpoints JSON after each frame and reuses only matching, checksummed completed predictions when rerun in the **same surviving Colab VM**. A runtime reset can erase temporary storage. You can interrupt this cell, then run the summary/export cells to download whatever completed before ending the session. To retry specific cases in the same VM, enter their exact labels in `ONLY_CASES`; leave it empty for all ten.

In [ ]:
import time

ONLY_CASES = []  # Example: ['spread_depth_pro_cuda_float16']; empty means all ten.
unknown = set(ONLY_CASES) - {case['label'] for case in cases}
if unknown:
    raise ValueError(f'Unknown cases: {sorted(unknown)}')
selected_cases = [case for case in cases if not ONLY_CASES or case['label'] in ONLY_CASES]
progress_path = results_root / 'case_progress.json'
progress = json.loads(progress_path.read_text()) if progress_path.exists() else {}
for case in selected_cases:
    command = [str(python), str(project / 'scripts/part2/benchmark_models.py'),
               '--model', case['model'], '--device', 'cuda', '--precision', case['precision'],
               '--source-root', case['source_root'], '--checkpoint', case['checkpoint'],
               '--sample-manifest', str(manifest_path), '--output', case['output'],
               '--pred-root', case['pred_root']]
    for sequence in case['sequences']:
        command += ['--sequence', sequence]
    if case['domain'] is not None:
        command += ['--domain', case['domain']]
    if case['process_res'] is not None:
        command += ['--process-res', str(case['process_res'])]
    log_path = logs_root / (case['label'] + '.log')
    print('Starting', case['label'], flush=True)
    tick = time.perf_counter()
    with log_path.open('a') as log:
        log.write('\nCASE ATTEMPT ' + datetime.datetime.now(datetime.timezone.utc).isoformat() + '\n')
        log.flush()
        process = subprocess.run(command, cwd=project, env=run_env, stdout=log, stderr=subprocess.STDOUT)
    elapsed = time.perf_counter() - tick
    record = {'returncode': process.returncode, 'whole_process_seconds': elapsed,
              'command': command, 'log': str(log_path), 'finished_utc': datetime.datetime.now(datetime.timezone.utc).isoformat()}
    output_path = Path(case['output'])
    if output_path.is_file():
        benchmark = json.loads(output_path.read_text())
        record['benchmark_status'] = benchmark.get('status')
        record['frames_ok'] = sum(row.get('status') == 'ok' for row in benchmark.get('frames', []))
        record['frames_recorded'] = len(benchmark.get('frames', []))
    progress.setdefault(case['label'], []).append(record)
    temporary = progress_path.with_suffix('.tmp')
    temporary.write_text(json.dumps(progress, indent=2))
    temporary.replace(progress_path)
    print('Finished', case['label'], 'returncode', process.returncode,
          f'wall={elapsed:.1f}s', 'frames_ok=', record.get('frames_ok', 0), flush=True)
    if process.returncode != 0:
        print('Failure retained; next case will still run. Last log lines:')
        print('\n'.join(log_path.read_text(errors='replace').splitlines()[-15:]))
print('Case loop finished. Continue to summary/export even when some cases failed.')

## 9. Build the auditable comparison table

Tables keep device, hardware, precision, checkpoint/domain and processing configuration distinct. Partial or failed cases remain visible and must not be treated as complete rankings. Raw GT errors measure metric quality; median-scaled errors diagnose shape after removing a per-frame scale. KITTI has no depth-quality score.

In [ ]:
summary_root = results_root / 'summary'
subprocess.run([str(python), str(project / 'scripts/part2/summarize_model_benchmarks.py'),
                '--input-root', str(benchmark_root), '--output-root', str(summary_root)],
               cwd=project, env=run_env, check=True)
from IPython.display import Markdown, display
display(Markdown((summary_root / 'comparison.md').read_text()))

## 10. View GPU-only tradeoffs and shared-scale depth maps

The scatter plots compare completed configurations on this GPU only. The KITTI panel reports inference time because depth GT is absent. Contact sheets show the first selected image of each sequence, with one common metre scale for all model predictions and GT in that sheet; values above the displayed range are clipped **only for visualization**. Missing/failed predictions are labelled.

In [ ]:
plot_code = r"""
import json, math, sys
from pathlib import Path
import cv2
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

project, results = map(Path, sys.argv[1:])
cases = json.loads((results/'case_plan.json').read_text())
manifest = json.loads((project/'sample_manifest.json').read_text())
figures = results/'figures'
figures.mkdir(exist_ok=True)
loaded = {}
for case in cases:
    path = Path(case['output'])
    if path.is_file():
        loaded[case['label']] = json.loads(path.read_text())

def label(case):
    name = {'depth_pro':'Depth Pro','da3_metric_large':'DA3 Metric L',
            'dav2_small':'V2 Small','dav2_base':'V2 Base','dav2_large':'V2 Large'}[case['model']]
    return name + (' FP16' if case['precision']=='float16' else ' FP32')

fig, axes = plt.subplots(1, 3, figsize=(19, 5))
for ax, sequence in zip(axes, ('replica_room0','tum_fr3','kitti06')):
    points=[]
    for case in cases:
        document = loaded.get(case['label'])
        if document is None or sequence not in case['sequences']:
            continue
        runtime = document.get('signature',{}).get('runtime',{})
        if not str(runtime.get('device','')).startswith('cuda'):
            continue
        rows = [row for row in document.get('frames',[]) if row.get('sequence')==sequence]
        if len(rows)!=3 or any(row.get('status')!='ok' for row in rows):
            continue
        seconds = float(np.mean([row['inference_seconds'] for row in rows]))
        if sequence=='kitti06':
            points.append((label(case),seconds,None))
        elif all(row.get('raw') and row['raw'].get('prediction_coverage')==1 for row in rows):
            points.append((label(case),seconds,float(np.mean([row['raw']['abs_rel'] for row in rows]))))
    if not points:
        ax.text(.5,.5,'No complete GPU cases',ha='center',va='center',transform=ax.transAxes)
    elif sequence=='kitti06':
        ax.barh([p[0] for p in points],[p[1] for p in points])
        ax.set_xlabel('Mean inference seconds/frame')
        ax.invert_yaxis()
    else:
        for title, seconds, error in points:
            ax.scatter(seconds,error)
            ax.annotate(title,(seconds,error),xytext=(4,4),textcoords='offset points',fontsize=8)
        ax.set_xscale('log')
        ax.set_xlabel('Mean inference seconds/frame (log scale)')
        ax.set_ylabel('Raw AbsRel (lower is better)')
        ax.grid(alpha=.25)
    ax.set_title(sequence + (' - no depth GT' if sequence=='kitti06' else ' - 3 frames, full GT-pixel coverage'))
fig.suptitle('Measured GPU configurations only; FP32 and FP16 remain separate')
fig.tight_layout()
fig.savefig(figures/'gpu_quality_time.png',dpi=150)
plt.close(fig)

for sequence in ('replica_room0','tum_fr3','kitti06'):
    item=next(row for row in manifest if row['sequence']==sequence)
    rgb=cv2.imread(str(project/item['rgb']))
    panels=[('RGB',cv2.cvtColor(rgb,cv2.COLOR_BGR2RGB),'rgb')]
    gt=None
    if item['depth_gt'] is not None:
        raw=cv2.imread(str(project/item['depth_gt']),cv2.IMREAD_UNCHANGED)
        gt=raw.astype(float)/item['gt_depth_factor']
        gt[raw==0]=np.nan
        panels.append(('Ground truth (m)',gt,'depth'))
    for case in cases:
        if sequence not in case['sequences']:
            continue
        document=loaded.get(case['label'],{})
        row=next((r for r in document.get('frames',[]) if r.get('sequence')==sequence and r.get('index')==item['index']),None)
        prediction=None
        if row and row.get('status')=='ok' and Path(row['prediction']).is_file():
            prediction=np.load(row['prediction'],allow_pickle=False)
        panels.append((label(case),prediction,'depth'))
    reference=gt
    if reference is None:
        arrays=[array[np.isfinite(array)&(array>0)] for _,array,kind in panels if kind=='depth' and array is not None]
        reference=np.concatenate(arrays) if arrays else np.array([1.])
    finite=reference[np.isfinite(reference)&(reference>0)]
    vmax=float(np.percentile(finite,98)) if finite.size else 1.
    vmax=max(vmax,.001)
    norm=Normalize(vmin=0,vmax=vmax,clip=True)
    rows=math.ceil(len(panels)/3)
    fig,axes=plt.subplots(rows,3,figsize=(15,rows*3.5),squeeze=False,layout='constrained')
    image_handle=None
    for ax,(title,array,kind) in zip(axes.flat,panels):
        if array is None:
            ax.text(.5,.5,'Unavailable / failed',ha='center',va='center',transform=ax.transAxes)
        elif kind=='rgb':
            ax.imshow(array)
        else:
            image_handle=ax.imshow(array,cmap='viridis',norm=norm)
        ax.set_title(title);ax.axis('off')
    for ax in list(axes.flat)[len(panels):]: ax.axis('off')
    if image_handle is not None:
        fig.colorbar(image_handle,ax=list(axes.flat),label='Camera-z depth (m)',shrink=.7)
    fig.suptitle(f'{sequence}, original frame {item["index"]}; shared range 0 to {vmax:.2f} m (display clipped)')
    fig.savefig(figures/f'depth_predictions_{sequence}.png',dpi=120)
    plt.close(fig)
print(json.dumps([str(path) for path in sorted(figures.glob('*.png'))]))
"""
figure_paths = json.loads(subprocess.check_output([str(python), '-c', plot_code, str(project), str(results_root)],
                                                 cwd=project, env=run_env, text=True))
from IPython.display import Image
for figure in figure_paths:
    display(Image(filename=figure))

## 11. Download measured results before ending the runtime

The ZIP contains benchmark JSON after each completed frame, predictions, failure logs, hardware/environment/checkpoint metadata, tables and figures. It excludes the input images, weights and cloned repositories. You can run this cell after interrupting the case loop to preserve partial work. Downloading does not extend a Colab allocation or make VM storage persistent.

In [ ]:
archive_path = Path('/content/depth_model_comparison_gpu_results.zip')
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for folder in (results_root, project / 'depth_maps/model_comparison_gpu'):
        if folder.is_dir():
            for file in sorted(folder.rglob('*')):
                if file.is_file():
                    archive.write(file, file.relative_to(project))
print('Download:', archive_path.name, f'({archive_path.stat().st_size / 1024**2:.1f} MiB)')
files.download(str(archive_path))

## Decide what to run on the full sequences

Use the measured table to choose a practical candidate with acceptable raw depth error, scale-adjusted shape error, prediction coverage and inference time. Compare FP16 against the same model's FP32 results rather than assuming equal quality. Check the contact sheets for boundary, sky and range failures; KITTI's depth accuracy remains unverified without GT.

Keep CPU and GPU measurements separate and retain their hardware labels. Extrapolated sequence runtimes are planning estimates, not measured full runs. The final coursework still needs full-sequence depth evaluation, temporal stability, keypoint uncertainty and fair SLAM comparisons. Return the downloaded results ZIP to compare these GPU measurements with the laptop CPU pilot.

**Validation of this notebook:** source cells, embedded child-process scripts and notebook structure are checked locally. Actual Colab allocation, installation, model memory fit, GPU inference and browser upload/download require your Colab session and have not been executed here.